[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/04_layernorm.ipynb)

# 🟡 Medium: Implement LayerNorm

Implement **Layer Normalization** from scratch.

$$\text{LayerNorm}(x) = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

where $\mu$ and $\sigma^2$ are computed over the **last dimension**.

### Signature
```python
def my_layer_norm(
    x: torch.Tensor,      # input
    gamma: torch.Tensor,   # scale (same size as last dim)
    beta: torch.Tensor,    # shift (same size as last dim)
    eps: float = 1e-5
) -> torch.Tensor:
    ...
```

### Rules
- Do **NOT** use `F.layer_norm` or `torch.nn.LayerNorm`
- Normalize over the last dimension only
- Must support autograd

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch

In [3]:
# ✏️ YOUR IMPLEMENTATION HERE

def my_layer_norm(x, gamma, beta, eps=1e-5):
    # 1. keepdim=True 해야 원본과 연산 시 브로드캐스팅 정상 실행
    mean = x.mean(dim=-1, keepdim=True)

    # PyTorch의 LayerNorm은 모집단 분산(biased variance, N으로 나눔)을 사용하므로 unbiased=False로 설정
    var = x.var(dim=-1, unbiased=False, keepdim=True)

    # 2. 정규화 (Normalization)
    x_normalized = (x - mean) / torch.sqrt(var + eps)

    # 3. 스케일(gamma) 및 시프트(beta) 적용
    return gamma * x_normalized + beta

In [4]:
# 🧪 Debug
x = torch.randn(2, 8)
gamma = torch.ones(8)
beta = torch.zeros(8)

out = my_layer_norm(x, gamma, beta)
ref = torch.nn.functional.layer_norm(x, [8], gamma, beta)

print("Your output mean:", out.mean(dim=-1))   # should be ~0
print("Your output std: ", out.std(dim=-1))     # should be ~1
print("Match ref?      ", torch.allclose(out, ref, atol=1e-4))

Your output mean: tensor([-1.4901e-08, -3.7253e-09])
Your output std:  tensor([1.0690, 1.0690])
Match ref?       True


In [5]:
# ✅ SUBMIT
from torch_judge import check
check("layernorm")


🧪 Testing: Implement LayerNorm (Medium)
──────────────────────────────────────────────────
  ✅ [1/3] Shape and basic behavior (2.1ms)
  ✅ [2/3] With learned parameters (0.9ms)
  ✅ [3/3] Gradient flow (20.0ms)
──────────────────────────────────────────────────
  🎉 All 3 tests passed! (23.0ms total)
  Progress saved. Run status() to see your dashboard.

